# multi_strat_010

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (157).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 12 |
| Detected functions | calculate_ohlc4, initialize_kalman_filter, apply_kalman_filter, backtest, backtest_all_tickers, calculate_ohlc4, initialize_kalman_filter, backtest, plot_ohlc4_vs_kalman, plot_balance_over_time, plot_cumulative_pnl, plot_trade_actions |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install numpy pandas matplotlib plotly pykalman


In [ ]:
import pandas as pd
path = 'drive/My Drive/ipynb'


In [ ]:
# Read the CSV file
data = pd.read_csv(path + '/HVStocks/merged_file.csv')  # Use forward slashes

# Display the DataFrame
data


In [ ]:
import pandas as pd

# Convert 'datetime' column to datetime type if not already
data['datetime'] = pd.to_datetime(data['date'])

# Filter for rows where the time is between 9:15 and 15:30
data = data[(data['datetime'].dt.time >= pd.to_datetime('09:15').time()) &
            (data['datetime'].dt.time <= pd.to_datetime('15:31').time())]

# Drop rows with NaN or 0 values in specified columns
data = data.dropna(subset=['open', 'high', 'low', 'close', 'volume'])
data = data[(data['open'] != 0) & (data['high'] != 0) & (data['low'] != 0) & (data['close'] != 0) & (data['volume'] != 0)]

# Display the cleaned DataFrame
print(data)


In [ ]:
import pandas as pd
import numpy as np
from pykalman import KalmanFilter
from tqdm.notebook import tqdm

def calculate_ohlc4(data):
    """
    Calculate the average of open, high, low, and close prices (OHLC4).
    """
    data['OHLC4'] = (data['open'] + data['high'] + data['low'] + data['close']) / 4
    return data

def initialize_kalman_filter(process_noise, measurement_noise):
    """
    Initialize the Kalman Filter with specified noise parameters.
    """
    kf = KalmanFilter(
        initial_state_mean=0,
        initial_state_covariance=1,
        transition_matrices=[1],
        observation_matrices=[1],
        transition_covariance=process_noise,
        observation_covariance=measurement_noise
    )
    return kf

def apply_kalman_filter(data, kf):
    """
    Apply the Kalman Filter to the OHLC4 column and add a Kalman_Price column.
    """
    data['Kalman_Price'], _ = kf.filter(data['OHLC4'].values)
    return data

def backtest(data, ticker, kf, tp_ratio=0.015, sl_ratio=0.005, brokerage=0.001, closing_time="15:28", fixed_capital=100000):
    """
    Run the backtest strategy using Kalman Filter and OHLC4 data for a specific ticker.
    """
    # Ensure the 'date' column is datetime format
    data['date'] = pd.to_datetime(data['date'])

    # Apply Kalman Filter
    data = apply_kalman_filter(data, kf)

    # Initialize backtest variables
    balance = fixed_capital
    position = None
    entry_price = 0
    entry_time = None
    trade_log = []

    # Backtest Loop
    for i in range(1, len(data)):
        current_time = data['date'].iloc[i].strftime("%H:%M")
        current_price = data['close'].iloc[i]
        kalman_price = data['Kalman_Price'].iloc[i]

        # Check for open trades and update balance if TP or SL is hit
        if position == "long":
            tp_price = entry_price * (1 + tp_ratio)
            sl_price = entry_price * (1 - sl_ratio)
            if current_price >= tp_price or current_price <= sl_price:
                position_size = fixed_capital / entry_price
                pnl = (current_price - entry_price) * position_size * (1 - brokerage)
                trade_log.append((data['date'].iloc[i], "Exit Long", ticker, entry_price, current_price, entry_time, data['date'].iloc[i], pnl, balance + pnl))
                balance += pnl
                position = None
        elif position == "short":
            tp_price = entry_price * (1 - tp_ratio)
            sl_price = entry_price * (1 + sl_ratio)
            if current_price <= tp_price or current_price >= sl_price:
                position_size = fixed_capital / entry_price
                pnl = (entry_price - current_price) * position_size * (1 - brokerage)
                trade_log.append((data['date'].iloc[i], "Exit Short", ticker, entry_price, current_price, entry_time, data['date'].iloc[i], pnl, balance + pnl))
                balance += pnl
                position = None

        # Close all open trades at the specified time
        if current_time == closing_time and position is not None:
            position_size = fixed_capital / entry_price
            pnl = (current_price - entry_price) * position_size * (1 - brokerage) if position == "long" else (entry_price - current_price) * position_size * (1 - brokerage)
            trade_log.append((data['date'].iloc[i], f"Close {position.capitalize()}", ticker, entry_price, current_price, entry_time, data['date'].iloc[i], pnl, balance + pnl))
            balance += pnl
            position = None

        # Check for new signals if no position is open
        if position is None:
            if current_price > kalman_price:
                position = "long"
                entry_price = current_price
                entry_time = data['date'].iloc[i]
                trade_log.append((data['date'].iloc[i], "Enter Long", ticker, entry_price, None, entry_time, None, None, balance))
            elif current_price < kalman_price:
                position = "short"
                entry_price = current_price
                entry_time = data['date'].iloc[i]
                trade_log.append((data['date'].iloc[i], "Enter Short", ticker, entry_price, None, entry_time, None, None, balance))

    # Final Output
    trade_df = pd.DataFrame(trade_log, columns=["DateTime", "Action", "Ticker", "Entry Price", "Exit Price", "Entry Time", "Exit Time", "PnL", "Balance"])
    return trade_df, balance

def backtest_all_tickers(data, process_noise=1e-7, measurement_noise=0.3):
    """
    Apply the backtesting strategy for all tickers in the dataset with a progress bar.
    """
    tickers = data['Ticker'].unique()
    all_trade_logs = []
    kf = initialize_kalman_filter(process_noise, measurement_noise)

    for ticker in tqdm(tickers, desc="Backtesting tickers"):
        ticker_data = data[data['Ticker'] == ticker].copy()
        ticker_data = calculate_ohlc4(ticker_data)
        trade_log, _ = backtest(ticker_data, ticker, kf)
        all_trade_logs.append(trade_log)

    return pd.concat(all_trade_logs, ignore_index=True)

# Example Usage
# Ensure `data` is a DataFrame with columns: date, open, high, low, close, Ticker
trade_logs = backtest_all_tickers(data)

# Save to CSV
trade_logs.to_csv("trade_log_all_tickers.csv", index=False)


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pykalman import KalmanFilter

# Calculate OHLC4 (average of OHLC)
data['OHLC4'] = (data['open'] + data['high'] + data['low'] + data['close']) / 4

# Kalman Filter parameters
process_noise = 1  # Tune this parameter
measurement_noise = 30  # Tune this parameter
kf = KalmanFilter(
    initial_state_mean=0,
    initial_state_covariance=1,
    transition_matrices=[1],
    observation_matrices=[1],
    transition_covariance=process_noise,
    observation_covariance=measurement_noise
)

# Apply Kalman Filter to OHLC4
data['Kalman_Price'], _ = kf.filter(data['OHLC4'].values)

# Plotting with Plotly
fig = go.Figure()

# Add Close Price
fig.add_trace(go.Scatter(x=np.arange(len(data)), y=data['close'], mode='lines', name='Close Price', line=dict(color='blue', width=1)))

# Add Kalman Filter Price
fig.add_trace(go.Scatter(x=np.arange(len(data)), y=data['Kalman_Price'], mode='lines', name='Kalman Filter Price', line=dict(color='green', width=2)))

# Layout settings
fig.update_layout(
    title='Close Price and Kalman Filter Price',
    xaxis_title='Index',
    yaxis_title='Price',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=600
)

# Show plot
fig.show()


In [ ]:
import pandas as pd
import numpy as np
from pykalman import KalmanFilter

def calculate_ohlc4(data):
    """
    Calculate the average of open, high, low, and close prices.
    """
    data['OHLC4'] = (data['open'] + data['high'] + data['low'] + data['close']) / 4
    return data

def initialize_kalman_filter(process_noise, measurement_noise):
    """
    Initialize the Kalman Filter with specified noise parameters.
    """
    kf = KalmanFilter(
        initial_state_mean=0,
        initial_state_covariance=1,
        transition_matrices=[1],
        observation_matrices=[1],
        transition_covariance=process_noise,
        observation_covariance=measurement_noise
    )
    return kf

def backtest(data, process_noise=1e-7, measurement_noise=0.3, tp_ratio=0.015, sl_ratio=0.005, brokerage=0.001, closing_time="15:28", fixed_capital=100000):
    """
    Run the backtest strategy using Kalman Filter and OHLC4 data.
    """
    # Ensure the 'date' column is datetime format
    data['date'] = pd.to_datetime(data['date'])

    # Step 1: Calculate OHLC4 and apply Kalman Filter
    data = calculate_ohlc4(data)
    kf = initialize_kalman_filter(process_noise, measurement_noise)
    data['Kalman_Price'], _ = kf.filter(data['OHLC4'].values)

    # Step 2: Initialize backtest variables
    balance = fixed_capital
    position = None
    entry_price = 0
    entry_time = None
    trade_log = []

    # Step 3: Backtest Loop
    for i in range(1, len(data)):
        current_time = data['date'].iloc[i].strftime("%H:%M")
        current_price = data['close'].iloc[i]
        kalman_price = data['Kalman_Price'].iloc[i]

        # Check for open trades and update balance if TP or SL is hit
        if position == "long":
            tp_price = entry_price * (1 + tp_ratio)
            sl_price = entry_price * (1 - sl_ratio)
            if current_price >= tp_price or current_price <= sl_price:
                position_size = fixed_capital / entry_price
                pnl = (current_price - entry_price) * position_size * (1 - brokerage)
                trade_log.append((data['date'].iloc[i], "Exit Long", entry_price, current_price, entry_time, data['date'].iloc[i], pnl, balance + pnl))
                balance += pnl
                position = None
        elif position == "short":
            tp_price = entry_price * (1 - tp_ratio)
            sl_price = entry_price * (1 + sl_ratio)
            if current_price <= tp_price or current_price >= sl_price:
                position_size = fixed_capital / entry_price
                pnl = (entry_price - current_price) * position_size * (1 - brokerage)
                trade_log.append((data['date'].iloc[i], "Exit Short", entry_price, current_price, entry_time, data['date'].iloc[i], pnl, balance + pnl))
                balance += pnl
                position = None

        # Close all open trades at the specified time
        if current_time == closing_time and position is not None:
            position_size = fixed_capital / entry_price
            pnl = (current_price - entry_price) * position_size * (1 - brokerage) if position == "long" else (entry_price - current_price) * position_size * (1 - brokerage)
            trade_log.append((data['date'].iloc[i], f"Close {position.capitalize()}", entry_price, current_price, entry_time, data['date'].iloc[i], pnl, balance + pnl))
            balance += pnl
            position = None

        # Check for new signals if no position is open
        if position is None:
            if current_price > kalman_price:
                position = "long"
                entry_price = current_price
                entry_time = data['date'].iloc[i]
                trade_log.append((data['date'].iloc[i], "Enter Long", entry_price, None, entry_time, None, None, balance))
            elif current_price < kalman_price:
                position = "short"
                entry_price = current_price
                entry_time = data['date'].iloc[i]
                trade_log.append((data['date'].iloc[i], "Enter Short", entry_price, None, entry_time, None, None, balance))

        # Close current position and reverse if an opposite signal is generated
        elif position == "long" and current_price < kalman_price:
            position_size = fixed_capital / entry_price
            pnl = (current_price - entry_price) * position_size * (1 - brokerage)
            trade_log.append((data['date'].iloc[i], "Exit Long for Short", entry_price, current_price, entry_time, data['date'].iloc[i], pnl, balance + pnl))
            balance += pnl
            position = "short"
            entry_price = current_price
            entry_time = data['date'].iloc[i]
            trade_log.append((data['date'].iloc[i], "Enter Short", entry_price, None, entry_time, None, None, balance))
        elif position == "short" and current_price > kalman_price:
            position_size = fixed_capital / entry_price
            pnl = (entry_price - current_price) * position_size * (1 - brokerage)
            trade_log.append((data['date'].iloc[i], "Exit Short for Long", entry_price, current_price, entry_time, data['date'].iloc[i], pnl, balance + pnl))
            balance += pnl
            position = "long"
            entry_price = current_price
            entry_time = data['date'].iloc[i]
            trade_log.append((data['date'].iloc[i], "Enter Long", entry_price, None, entry_time, None, None, balance))

    # Step 4: Final Output and Return Results
    trade_df = pd.DataFrame(trade_log, columns=["DateTime", "Action", "Entry Price", "Exit Price", "Entry Time", "Exit Time", "PnL", "Balance"])
    return trade_df, balance

# Example Usage
# Ensure `data` is a DataFrame with columns: date, open, high, low, close
trade_df, final_balance = backtest(data)

# print(f"Final Balance: {final_balance}")
trade_df.to_csv("trade_log.csv", index=False)


In [ ]:
trade_df = trade_logs
trade_df

In [ ]:
import plotly.graph_objects as go

# Extract data for plotting
cumulative_pnl = trade_df['Balance']
trade_dates = trade_df['DateTime']  # Assuming trades are closed on the same day

# Create the plot
fig = go.Figure()

# Add Cumulative PnL as a line plot
fig.add_trace(go.Scatter(
    x=trade_dates,
    y=cumulative_pnl,
    mode='lines+markers',
    name='Cumulative PnL',
    line=dict(color='blue', width=2),
    marker=dict(size=6)
))

# Add zero line for reference
fig.add_trace(go.Scatter(
    x=trade_dates,
    y=[0] * len(cumulative_pnl),
    mode='lines',
    name='Zero Line',
    line=dict(color='gray', width=1, dash='dash')
))

# Update layout
fig.update_layout(
    title='Cumulative PnL Over Time',
    xaxis_title='Trade Date',
    yaxis_title='Cumulative PnL',
    template='plotly_white',
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

# Show plot
fig.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure seaborn uses a modern style
sns.set(style="whitegrid")

# 1. OHLC4 and Kalman Filter Price
def plot_ohlc4_vs_kalman(data):
    plt.figure(figsize=(12, 6))
    plt.plot(data['date'], data['OHLC4'], label='OHLC4', color='blue', alpha=0.7)
    plt.plot(data['date'], data['Kalman_Price'], label='Kalman Filter Price', color='orange', linestyle='--')
    plt.title("OHLC4 vs Kalman Filter Price")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.show()

# 2. Balance Over Time
def plot_balance_over_time(trade_df):
    plt.figure(figsize=(12, 6))
    plt.plot(trade_df['DateTime'], trade_df['Balance'], label='Balance', color='green')
    plt.title("Balance Over Time")
    plt.xlabel("Date")
    plt.ylabel("Balance")
    plt.legend()
    plt.show()

# 3. Cumulative PnL Over Time
def plot_cumulative_pnl(trade_df):
    trade_df['Cumulative PnL'] = trade_df['PnL'].cumsum()
    plt.figure(figsize=(12, 6))
    plt.plot(trade_df['DateTime'], trade_df['Cumulative PnL'], label='Cumulative PnL', color='purple')
    plt.title("Cumulative PnL Over Time")
    plt.xlabel("Date")
    plt.ylabel("Cumulative PnL")
    plt.legend()
    plt.show()

# 4. Trade Actions Over Time
def plot_trade_actions(trade_df):
    plt.figure(figsize=(12, 6))
    plt.scatter(trade_df[trade_df['Action'].str.contains("Enter")]['DateTime'],
                trade_df[trade_df['Action'].str.contains("Enter")]['Entry Price'],
                color='blue', label='Entry', marker='^', alpha=0.7)
    plt.scatter(trade_df[trade_df['Action'].str.contains("Exit")]['DateTime'],
                trade_df[trade_df['Action'].str.contains("Exit")]['Exit Price'],
                color='red', label='Exit', marker='v', alpha=0.7)
    plt.title("Trade Entry and Exit Points")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.show()

# 5. Profit and Loss Distribution
def plot_pnl_distribution(trade_df):
    plt.figure(figsize=(12, 6))
    sns.histplot(trade_df['PnL'], bins=50, kde=True, color='skyblue')
    plt.title("PnL Distribution")
    plt.xlabel("PnL")
    plt.ylabel("Frequency")
    plt.show()

# 6. Monthly Performance (assuming 'DateTime' is datetime type)
def plot_monthly_performance(trade_df):
    trade_df['Month'] = trade_df['DateTime'].dt.to_period('M')
    monthly_pnl = trade_df.groupby('Month')['PnL'].sum()
    plt.figure(figsize=(12, 6))
    monthly_pnl.plot(kind='bar', color='goldenrod')
    plt.title("Monthly Performance")
    plt.xlabel("Month")
    plt.ylabel("PnL")
    plt.show()

# 7. Entry and Exit Prices with Kalman Filter
def plot_trades_with_kalman(data, trade_df):
    plt.figure(figsize=(12, 6))
    plt.plot(data['date'], data['Kalman_Price'], label='Kalman Filter Price', color='orange', linestyle='--')
    plt.scatter(trade_df[trade_df['Action'].str.contains("Enter")]['DateTime'],
                trade_df[trade_df['Action'].str.contains("Enter")]['Entry Price'],
                color='blue', label='Entry', marker='^', alpha=0.7)
    plt.scatter(trade_df[trade_df['Action'].str.contains("Exit")]['DateTime'],
                trade_df[trade_df['Action'].str.contains("Exit")]['Exit Price'],
                color='red', label='Exit', marker='v', alpha=0.7)
    plt.title("Trades with Kalman Filter")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.show()

# 8. Win vs Loss Count
def plot_win_loss(trade_df):
    trade_df['Win'] = trade_df['PnL'] > 0
    win_loss_counts = trade_df['Win'].value_counts()
    plt.figure(figsize=(6, 6))
    win_loss_counts.plot(kind='pie', labels=['Wins', 'Losses'], autopct='%1.1f%%', startangle=90, colors=['green', 'red'])
    plt.title("Win vs Loss Percentage")
    plt.ylabel("")
    plt.show()

# Example Usage (Uncomment the plots you want to use)
# Ensure `data` and `trade_df` are available from the backtest
plot_ohlc4_vs_kalman(data)
plot_balance_over_time(trade_df)
plot_cumulative_pnl(trade_df)
plot_trade_actions(trade_df)
plot_pnl_distribution(trade_df)
plot_monthly_performance(trade_df)
plot_trades_with_kalman(data, trade_df)
plot_win_loss(trade_df)


In [ ]:
import pandas as pd
import numpy as np
from pykalman import KalmanFilter
from tqdm import tqdm
from joblib import Parallel, delayed

def calculate_ohlc4(data):
    """
    Calculate the average of open, high, low, and close prices.
    """
    return (data['open'] + data['high'] + data['low'] + data['close']) / 4

def initialize_kalman_filter(process_noise, measurement_noise):
    """
    Initialize the Kalman Filter with specified noise parameters.
    """
    kf = KalmanFilter(
        initial_state_mean=0,
        initial_state_covariance=1,
        transition_matrices=[1],
        observation_matrices=[1],
        transition_covariance=process_noise,
        observation_covariance=measurement_noise
    )
    return kf

def backtest(data, process_noise, measurement_noise, tp_ratio, sl_ratio, brokerage=0.001, closing_time="15:28", fixed_capital=100000):
    """
    Run the backtest strategy using Kalman Filter and OHLC4 data.
    """
    # Step 1: Calculate OHLC4 and apply Kalman Filter
    ohlc4 = calculate_ohlc4(data)
    kf = initialize_kalman_filter(process_noise, measurement_noise)
    kalman_price = kf.filter(ohlc4.values)[0]

    # Step 2: Initialize backtest variables
    balance = fixed_capital
    position = None
    entry_price = 0
    entry_time = None
    trade_log = []

    # Precompute all trades
    for i in range(1, len(data)):
        current_time = pd.Timestamp(data.index[i]).strftime("%H:%M")
        current_price = data['close'].iloc[i]
        kf_price = kalman_price[i]

        # Entry signal
        if position is None:
            if current_price > kf_price:
                position = "long"
                entry_price = current_price
                entry_time = data.index[i]
                trade_log.append((data.index[i], "Enter Long", entry_price, None, entry_time, None, None, balance))
            elif current_price < kf_price:
                position = "short"
                entry_price = current_price
                entry_time = data.index[i]
                trade_log.append((data.index[i], "Enter Short", entry_price, None, entry_time, None, None, balance))

        # Exit logic for TP, SL, or closing time
        elif position == "long":
            tp_price = entry_price * (1 + tp_ratio)
            sl_price = entry_price * (1 - sl_ratio)
            if current_price >= tp_price or current_price <= sl_price or current_time == closing_time:
                position_size = fixed_capital / entry_price
                pnl = (current_price - entry_price) * position_size * (1 - brokerage)
                trade_log.append((data.index[i], "Exit Long", entry_price, current_price, entry_time, data.index[i], pnl, balance + pnl))
                balance += pnl
                position = None

        elif position == "short":
            tp_price = entry_price * (1 - tp_ratio)
            sl_price = entry_price * (1 + sl_ratio)
            if current_price <= tp_price or current_price >= sl_price or current_time == closing_time:
                position_size = fixed_capital / entry_price
                pnl = (entry_price - current_price) * position_size * (1 - brokerage)
                trade_log.append((data.index[i], "Exit Short", entry_price, current_price, entry_time, data.index[i], pnl, balance + pnl))
                balance += pnl
                position = None

        # Reverse position if an opposite signal occurs
        elif position == "long" and current_price < kf_price:
            position_size = fixed_capital / entry_price
            pnl = (current_price - entry_price) * position_size * (1 - brokerage)
            trade_log.append((data.index[i], "Exit Long for Short", entry_price, current_price, entry_time, data.index[i], pnl, balance + pnl))
            balance += pnl
            position = "short"
            entry_price = current_price
            entry_time = data.index[i]
            trade_log.append((data.index[i], "Enter Short", entry_price, None, entry_time, None, None, balance))

        elif position == "short" and current_price > kf_price:
            position_size = fixed_capital / entry_price
            pnl = (entry_price - current_price) * position_size * (1 - brokerage)
            trade_log.append((data.index[i], "Exit Short for Long", entry_price, current_price, entry_time, data.index[i], pnl, balance + pnl))
            balance += pnl
            position = "long"
            entry_price = current_price
            entry_time = data.index[i]
            trade_log.append((data.index[i], "Enter Long", entry_price, None, entry_time, None, None, balance))

    # Return final results
    trade_df = pd.DataFrame(trade_log, columns=["DateTime", "Action", "Entry Price", "Exit Price", "Entry Time", "Exit Time", "PnL", "Balance"])
    return trade_df, balance

def run_robustness_backtest(tp_ratio, sl_ratio, process_noise, measurement_noise, data, fixed_capital):
    """
    Run a single backtest with specific parameters and return results.
    """
    trade_df, final_balance = backtest(data, process_noise, measurement_noise, tp_ratio, sl_ratio, fixed_capital=fixed_capital)
    return {
        'TP Ratio': tp_ratio,
        'SL Ratio': sl_ratio,
        'Process Noise': process_noise,
        'Measurement Noise': measurement_noise,
        'Final Balance': final_balance,
        'Total Trades': len(trade_df),
    }

def robustness_testing(data, tp_ratios, sl_ratios, process_noises, measurement_noises, fixed_capital=100000):
    """
    Run robustness tests by varying different parameters and track the results.
    """
    # Run parallel backtesting
    results = Parallel(n_jobs=-1)(
        delayed(run_robustness_backtest)(tp_ratio, sl_ratio, process_noise, measurement_noise, data, fixed_capital)
        for tp_ratio in tp_ratios
        for sl_ratio in sl_ratios
        for process_noise in process_noises
        for measurement_noise in measurement_noises
    )

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    return results_df

# Define parameter ranges for robustness testing
tp_ratios = [0.01, 0.015, 0.02]
sl_ratios = [0.0025, 0.005, 0.01, 0.015]
process_noises = [0.00001, 0.0003, 0.00005, 0.00007, 0.0001]
measurement_noises = [0.008, 0.01, 0.03, 0.05, 0.07, 0.1]

# Run robustness testing
print("Running robustness testing...")
results_df = robustness_testing(data, tp_ratios, sl_ratios, process_noises, measurement_noises)

# Display the results
print(results_df)

# Save the results to CSV
results_df.to_csv("robustness_testing_results.csv", index=False)
